# CARLA RL Comparison — TD3 Training

**Algorithm:** TD3  |  **Shared environment:** `common.CarlaDrivingEnv`  |  **Repo:** `2005-ab/carla-rl-comparison`

**Owner:** Nemo (TD3). Teammates Abhi (SAC) and Rasika (PPO) use the same shared environment.

This notebook trains and evaluates a TD3 agent on the **same** Gymnasium environment,
reward function, observation/action spaces, routes, and evaluation protocol used by the SAC,
TD3, and PPO baselines, so results are directly comparable.

> Run cells top-to-bottom. The **smoke-test** cell runs a short training automatically; the
> **full training** cell is guarded by `RUN_FULL_TRAINING = False` and will NOT start a long run
> unless you explicitly enable it.

## 1. Dependencies & workspace setup

Installs/imports everything and (in Colab) clones the shared repo so the `common` package is
importable. The CARLA Python API version must match the remote CARLA server version.

In [ ]:
# === Dependencies & workspace setup ===
import os, sys, time, shutil

# Detect Google Colab.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    # Clone the shared repo (public) and install its Colab requirements.
    repo_url = "https://github.com/2005-ab/carla-rl-comparison.git"
    repo_dir = "carla-rl-comparison"
    if not os.path.isdir(repo_dir):
        !git clone --depth 1 {repo_url} {repo_dir}
    os.chdir(repo_dir)
    !pip install -q -r requirements-colab.txt
    # Install the CARLA client matching your server version, e.g.:
    # !pip install carla==0.9.15
else:
    # Local execution: ensure the workspace root is importable.
    _ROOT = os.getcwd()
    if _ROOT not in sys.path:
        sys.path.insert(0, _ROOT)

import numpy as np
import torch
from stable_baselines3 import TD3

print("In Colab:", IN_COLAB)
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("NumPy:", np.__version__)

## 2. Configuration

Centralized parameters come from `common.config.EnvConfig`. Override host/port and seeds here.
All three notebooks use the **same** `EnvConfig`, reward weights, observation/action spaces,
and `EvaluationConfig` so the benchmark is fair.

In [ ]:
# === Configuration ===
from common.config import EnvConfig, SimulatorConfig, EvaluationConfig, RouteConfig, RewardConfig

# CARLA server endpoint (override as needed, e.g. for a remote/Colab host).
HOST = "localhost"
PORT = 2000
SEED = 42

# Build the shared environment configuration.
config = EnvConfig()
config.simulator = SimulatorConfig(host=HOST, port=PORT)
config.seed = SEED

# Keep results/checkpoints/models separate per algorithm but in shared folders.
ALGO = "TD3"          # used for filenames
MODELS_DIR = "models"
RESULTS_DIR = "results"
CKPT_DIR = os.path.join("checkpoints", ALGO.lower())
TENSORBOARD_DIR = "tensorboard_logs"
for d in (MODELS_DIR, RESULTS_DIR, CKPT_DIR, TENSORBOARD_DIR):
    os.makedirs(d, exist_ok=True)

# Persist the exact configuration used for this run (for reproducibility).
config.save_json(os.path.join(RESULTS_DIR, f"{ALGO.lower()}_config.json"))

eval_config = EvaluationConfig(n_eval_episodes=10, deterministic=True)
print("Env config:", config.to_dict())

## 3. CARLA connectivity check

Fails with a clear message if the `carla` package is missing, the server is unreachable, or the
client/server versions are incompatible.

In [ ]:
# === CARLA connectivity check ===
try:
    import carla
except ImportError as e:
    raise RuntimeError(
        "The 'carla' Python package is not installed. "
        "Install the version matching your CARLA server, e.g.  pip install carla==0.9.15"
    ) from e

client = carla.Client(HOST, PORT)
client.set_timeout(config.simulator.timeout)
try:
    world = client.get_world()
    carla_map = world.get_map()
except Exception as e:
    raise RuntimeError(
        f"Could not connect to CARLA at {HOST}:{PORT}. Is the simulator running? ({e})"
    ) from e

print("CARLA server version:", client.get_server_version())
print("Client API version   :", client.get_client_version())
print("Map                  :", carla_map.name)
print("Spawn points         :", len(carla_map.get_spawn_points()))
if client.get_server_version() != client.get_client_version():
    print("WARNING: client/server versions differ - behavior may be unstable.")

## 4. Build & validate the environment

Constructs the shared `CarlaDrivingEnv` against the live server and runs
`gymnasium.check_env` to confirm API compliance. A short manual rollout is also printed.

In [ ]:
# === Build & validate the environment ===
import warnings
from gymnasium.utils.env_checker import check_env
from common.carla_env import CarlaDrivingEnv

env = CarlaDrivingEnv(config=config, connect_on_init=True)
print("Connected to CARLA:", env.is_connected)
print("Observation space:", env.observation_space)
print("Action space    :", env.action_space)

# Validate the Gymnasium API contract on the real server.
with warnings.catch_warnings():
    warnings.simplefilter("error")
    check_env(env, skip_render_check=True)
print("gymnasium.check_env: PASSED")

obs, info = env.reset(seed=SEED)
print("Initial obs:", obs, "info dist_to_goal_m:", round(info["distance_to_goal_m"], 2))

## 5. Create the TD3 model

Hyperparameters are defined explicitly (no magic numbers). TensorBoard logs go to a shared
directory; per-algorithm checkpoints go to `checkpoints/<algo>/`.

In [ ]:
# === Create the model ===
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

from stable_baselines3 import TD3
from stable_baselines3.common.noise import NormalActionNoise
n_actions = env.action_space.shape[0]
action_noise = NormalActionNoise(
    mean=np.zeros(n_actions), sigma=0.1 * np.ones(n_actions)
)
model = TD3(
    "MlpPolicy", env,
    learning_rate=3e-4,
    buffer_size=100_000,
    learning_starts=1_000,
    batch_size=256,
    gamma=0.99,
    tau=0.005,
    policy_delay=2,
    action_noise=action_noise,
    verbose=1,
    tensorboard_log=TENSORBOARD_DIR,
    seed=SEED,
    device=device,
)

print(model)

## 6. Checkpointing callback

Saves a model snapshot every `SAVE_FREQ` steps so long runs are resumable.

In [ ]:
# === Checkpoint callback ===
from stable_baselines3.common.callbacks import CheckpointCallback

SAVE_FREQ = 5_000  # steps between checkpoints
checkpoint_cb = CheckpointCallback(
    save_freq=SAVE_FREQ,
    save_path=CKPT_DIR,
    name_prefix=ALGO.lower(),
    save_replay_buffer=True,
    save_vec_normalize=False,
)

## 7. Smoke-test training (short, runs automatically)

A quick sanity run to confirm training works end-to-end before committing to a long run.

In [ ]:
# === Smoke-test training ===
SMOKE_STEPS = 1_000
print(f"Smoke training {ALGO} for {SMOKE_STEPS} steps ...")
t0 = time.time()
model.learn(total_timesteps=SMOKE_STEPS, callback=checkpoint_cb, tb_log_name=ALGO, reset_num_timesteps=True)
print(f"Smoke training done in {time.time() - t0:.1f}s")

## 8. Full training (manual — does NOT auto-run)

Set `RUN_FULL_TRAINING = True` and run this cell only when you are ready for a long run.
Total steps and wall-clock time are recorded for the comparison report.

In [ ]:
# === Full training (manual) ===
RUN_FULL_TRAINING = False        # <- set True to actually train
TOTAL_STEPS = 50_000             # increase for real experiments

training_seconds = None
if RUN_FULL_TRAINING:
    print(f"Full training {ALGO} for {TOTAL_STEPS} steps ...")
    t0 = time.time()
    model.learn(total_timesteps=TOTAL_STEPS, callback=checkpoint_cb,
                tb_log_name=ALGO, reset_num_timesteps=True)
    training_seconds = time.time() - t0
    print(f"Full training done in {training_seconds:.1f}s")
else:
    print("Skipped: set RUN_FULL_TRAINING = True to start full training.")

## 9. Save the trained model

In [ ]:
# === Save model ===
model_path = os.path.join(MODELS_DIR, f"{ALGO.lower()}_carla")
model.save(model_path)
print("Saved model to:", model_path + ".zip")

## 10. Standardized evaluation

Uses `common.evaluation.evaluate_policy` so every algorithm exports the **same** CSV schema:
`<algo>_episodes.csv` (per-episode) and `<algo>_summary.csv` (aggregate metrics).

In [ ]:
# === Standardized evaluation ===
from common.evaluation import evaluate_policy, aggregate_results

summary, episodes_df = evaluate_policy(
    model=model,
    env=env,
    algorithm_name=ALGO,
    n_eval_episodes=eval_config.n_eval_episodes,
    deterministic=eval_config.deterministic,
    seeds=list(eval_config.eval_seeds),
    results_dir=RESULTS_DIR,
    training_wall_clock_sec=training_seconds,
)

print("=== Evaluation summary ===")
for k, v in summary.to_dict().items():
    print(f"  {k}: {v}")

# Combine all algorithm summaries into one comparison table when present.
try:
    combined = aggregate_results(RESULTS_DIR)
    print("Combined results:")
    print(combined.to_string(index=False))
except Exception as e:
    print("Could not aggregate:", e)

## 11. (Colab) Save artifacts to Google Drive

Mounts Drive and copies the model + CSVs into a folder under MyDrive. **No personal paths are
hardcoded** — the destination folder is built from the algorithm name. This cell only runs in
Colab.

In [ ]:
# === Google Drive save (Colab only) ===
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    drive_root = "/content/drive/MyDrive"
    dest = os.path.join(drive_root, "carla_rl_results", ALGO.lower())
    os.makedirs(dest, exist_ok=True)
    # Copy model.
    shutil.copy(model_path + ".zip", dest)
    # Copy result CSVs for this algorithm.
    for f in os.listdir(RESULTS_DIR):
        if f.lower().startswith(ALGO.lower()) and f.endswith(".csv"):
            shutil.copy(os.path.join(RESULTS_DIR, f), dest)
    print("Saved artifacts to:", dest)
else:
    print("Not in Colab; skipping Google Drive save.")

## 12. Load a saved model & run a demonstration

Reloads the model from disk and rolls out a couple of episodes to verify persistence.

In [ ]:
# === Load & demonstrate ===
from stable_baselines3 import TD3 as _AlgoCls

loaded = _AlgoCls.load(model_path, device=device)
print("Loaded model from:", model_path + ".zip")

for ep in range(2):
    obs, info = env.reset(seed=SEED + ep)
    done = False
    ep_reward = 0.0
    steps = 0
    while not done:
        action, _ = loaded.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        ep_reward += reward
        steps += 1
        done = terminated or truncated
    print(f"Demo ep {ep + 1}: reward={ep_reward:.2f}, steps={steps}, "
          f"goal={info['goal_reached']}, collision={info['collision']}, "
          f"route_completion={info['route_completion_pct']:.1f}%")

env.close()
print("Done.")

## 13. TensorBoard

Optional: launch TensorBoard to inspect training curves (run in a separate terminal, or use the
Colab notebook extension).

In [ ]:
# === TensorBoard (optional) ===
if IN_COLAB:
    %load_ext tensorboard
    %tensorboard --logdir {TENSORBOARD_DIR}
else:
    print(f"Run in a terminal:  tensorboard --logdir {TENSORBOARD_DIR}")